# Motifs — How Connected Neurons Behave Together

## What is a motif?

A **neural motif** is a small, recurring pattern of connectivity between neurons. Just as the same chord can appear in many different songs, the same small circuit patterns appear throughout the brain — in the cortex, cerebellum, brainstem, and sensory systems.

Understanding motifs is important because the *way neurons are connected* determines what a circuit can compute — not just the properties of individual neurons.

In this notebook we will build up from a **pair** of connected neurons to a **triplet**, and observe how connectivity shapes activity.

> 💡 As before: you do not need to understand the code. Focus on the sliders, the plots, and the questions. Write your answers directly into this notebook — you will submit it as a PDF at the end.



In [1]:
#@title Run the following to initialize lab environment.
!pip install "jupyter-book<2" scipy ipywidgets ipympl git+https://github.com/weirdoglh/stg-net.git -q

import matplotlib.pyplot as plt         # import matplotlib
from matplotlib.widgets import Slider
import numpy as np                      # import numpy
import ipywidgets as widgets            # interactive display

# Colab setting for widget
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# modeling library
from stg_net.neuron import LIF
from stg_net.input import Poisson_generator, Gaussian_generator, Current_injector
from stg_net.conn import Simulator
from stg_net.helper import plot_volt_trace

# setting for figures
fig_w, fig_h = 8, 6
my_fontsize = 18
my_params = {'axes.labelsize': my_fontsize,
          'axes.titlesize': my_fontsize,
          'figure.figsize': (fig_w, fig_h),
          'font.size': my_fontsize,
          'legend.fontsize': my_fontsize-4,
          'lines.markersize': 8.,
          'lines.linewidth': 2.,
          'xtick.labelsize': my_fontsize-2,
          'ytick.labelsize': my_fontsize-2}
plt.rcParams.update(my_params)
my_layout = widgets.Layout()

# Widget interaction
%matplotlib inline

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.6/82.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.2/83.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.9/455.9 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.0 MB/s eta 0:00:00
   ━━

## Part 1: An Excitatory-Inhibitory (E-I) Pair

The most fundamental motif in the brain is a pair of neurons — one **excitatory (E)** and one **inhibitory (I)**.

- **Excitatory neurons** increase the likelihood that their target will fire (positive connection weight)
- **Inhibitory neurons** decrease the likelihood that their target will fire (negative connection weight)


<center> <img src="https://github.com/michaelglh/STG/blob/master/figs/EI.png?raw=1" alt="EI motif" width="300"/></center>

In the simulation below:
- **Blue neuron (E)** receives constant input and may drive the red neuron
- **Red neuron (I)** can feed back onto the blue neuron

The sliders control:
- **J_ie** — how strongly E drives I (positive = excitatory input from E to I)
- **J_ei** — how strongly I feeds back onto E (negative = inhibitory feedback from I to E)

**Reading the plots:**
- **Raster plot** — each vertical tick is one spike. Blue = E neuron, Red = I neuron
- **Voltage trace** — the membrane potential of each neuron over time
- **Spike correlation** — how often the two neurons fire *at the same time* or with a fixed delay. A peak at 0ms means they tend to fire together (synchrony). A peak offset to one side means one reliably fires slightly before the other.

---

### Predict

*If neuron E excites neuron I (J_ie > 0), and neuron I inhibits neuron E (J_ei < 0), what do you think will happen to E's firing rate?*

**My prediction:**
```
[Write here]
```

*Do you think the two neurons will tend to fire at the same time, or at different times? Why?*

**My prediction:**
```
[Write here]
```


In [4]:
#@title Run the following to start E-I pair simulation { vertical-output: true }
# E-I motif
N = 2                   # number of neurons
T, dt = 5e2, 0.1        # simulation period(ms), step size(ms)
wt, dl = 1, 5.
rt = 65.

tonic_neuron = {'tau_m':20., 'a':0., 'tau_w':30., 'b':3., 'V_reset':-55.}
adapting_neuron = {'tau_m':20., 'a':0., 'tau_w':100., 'b':0.5, 'V_reset':-55.}
initburst_neuron = {'tau_m':10., 'a':0., 'tau_w':100., 'b':1., 'V_reset':-50.}
bursting_neuron = {'tau_m':5., 'a':0., 'tau_w':100., 'b':1., 'V_reset':-46.}
irregular_neuron = {'tau_m':10., 'a':-0.01, 'tau_w':50., 'b':1.2, 'V_reset':-46.}
transient_neuron = {'tau_m':5., 'a':0.05, 'tau_w':100., 'b':0.7, 'V_reset':-60.}
delayed_neuron = {'tau_m':5., 'a':-0.1, 'tau_w':100., 'b':1., 'V_reset':-60.}

neuron_params = {'tonic_neuron': tonic_neuron, 'adapting_neuron': adapting_neuron,
                 'initburst_neuron': initburst_neuron, 'bursting_neuron': bursting_neuron,
                 'irregular_neuron': irregular_neuron, 'transient_neuron': transient_neuron,
                 'delayed_neuron': delayed_neuron, 'my_neuron': tonic_neuron}

# updating parameters
def update_EI(Itype='Icur', neuron_E='tonic_neuron', neuron_I='tonic_neuron', J_ei=0., J_ie=0.):
    # simualtor
    h = Simulator(dt=dt)

    # network of neurons
    nrns = [LIF(sim=h) for _ in range(N)]

    nrns[0].update(neuron_params[neuron_E])
    nrns[1].update(neuron_params[neuron_I])

    # background noise
    if Itype == 'Icur':
        noises = [Current_injector(sim=h, rate=rt, start=int(T/dt*0.1), end=int(T/dt*0.9)) for _ in range(N)]
    elif Itype == 'Gaussian':
        noises = [Gaussian_generator(sim=h, mean=rt, std=rt) for _ in range(N)]
    elif Itype == 'Poisson':
        noises = [Poisson_generator(sim=h, rate=250) for _ in range(N)]
    else:
        print('Invalid input')
    for noise, nrn in zip(noises, nrns):
        nrn.connect(noise, {'ctype':'Static', 'weight':wt, 'delay':dl})

    # recurrent connections
    tps = [['Static']*N]*N
    con = np.array([[0., J_ei],
                    [J_ie, 0.]])
    dly = np.random.uniform(2., 5., (N,N))
    synspecs = [[{} for _ in range(N)] for _ in range(N)]
    for i in range(N):
        for j in range(N):
            synspecs[i][j] = {'ctype':tps[i][j], 'weight':con[i,j], 'delay':dly[i,j]}
    h.connect(nrns, nrns, synspecs)

    # simulation
    h.run(T)

    # coincidence
    binwindow = int(5.0/dt)
    spike_trains = [nrn.states for nrn in nrns]
    bin_spikes = [np.convolve(strain, np.ones(binwindow), 'same') for strain in spike_trains]
    deltats = np.linspace(-10., 10., 21)
    coins = []
    for delay in deltats:
        index = int(delay/dt)
        if index > 0:
            coins.append(np.dot(bin_spikes[0][:-index], bin_spikes[1][index:]))
        elif index < 0:
            coins.append(np.dot(bin_spikes[1][:index], bin_spikes[0][-index:]))
        else:
            coins.append(np.dot(bin_spikes[0], bin_spikes[1]))
    coins = np.array(coins)/np.sqrt(np.dot(bin_spikes[0], bin_spikes[0]))/np.sqrt(np.dot(bin_spikes[1], bin_spikes[1]))

    # visualize
    plt.clf()
    cs = ['b', 'r']
    plt.subplot(2,N,1)
    plt.title('raster')
    for nrn, c, l in zip(nrns, cs, range(N)):
        plt.eventplot(nrn.spikes['times'], lineoffsets=2*l, colors=c)
    plt.xlabel('Time(ms)')
    plt.yticks([0, 2], ['E', 'I'])
    plt.xlim([0., T])

    plt.subplot(2,N,2)
    plt.title('spike correlation')
    plt.plot(deltats, coins)
    plt.xlabel(r'$\delta t$(ms)')
    plt.ylabel('spike corr')
    plt.ylim([0, 1])

    # voltage trace
    for id, c in zip(range(N), cs):
        plt.subplot(2,N,id+N+1)
        if id == 0:
            plt.title('voltage trace')
        plt_par = {'dt':dt, 'range_t':np.arange(0., T, dt), 'V_th':nrns[id].V_th}
        plot_volt_trace(plt_par, nrns[id].v, np.array(nrns[id].spikes['times']), c=c)
        plt.xlim([0., T])

    plt.tight_layout()

widgets.interact(update_EI, Itype=['Icur', 'Gaussian', 'Poisson'],
                 neuron_E=neuron_params.keys(), neuron_I=neuron_params.keys(),
                 J_ie=widgets.FloatSlider(min=0.0, max=50.0, step=0.1, value=10.0, description='J_ie'),
                 J_ei=widgets.FloatSlider(min=-50.0, max=0.0, step=0.1, value=-10.0, description='J_ei'));

interactive(children=(Dropdown(description='Itype', options=('Icur', 'Gaussian', 'Poisson'), value='Icur'), Dr…

### Explain — E-I Pair

**Start with no connections (J_ie = 0, J_ei = 0)**

*Describe what both neurons are doing. Are they correlated at all?*

**My answer:**
```
[Write here]
```

---

**Now increase J_ie (E drives I)**

*Set J_ie to around 10. What happens to the red (I) neuron's firing?*

**My answer:**
```
[Write here]
```

*Look at the spike correlation plot. Does the peak shift, and which direction? What does this tell you about the timing relationship?*

**My answer:**
```
[Write here]
```

---

**Now add inhibitory feedback (J_ei negative)**

*Keep J_ie at 10 and set J_ei to around -10. What happens to the blue (E) neuron's firing rate?*

**My answer:**
```
[Write here]
```

*This is called a **feedback inhibition** circuit. Can you think of a reason why the brain might use this design? What could it be useful for?*

> Hint: think about what happens when a signal gets too strong

**My answer:**
```
[Write here]
```

---

**Exploring input type**

*Switch from Icur to Poisson input. The neurons now fire irregularly on their own. Can you find a connection weight that makes them fire more periodically (rhythmically)?*

*What values worked? What does this suggest about the role of inhibitory feedback?*

**My answer:**
```
[Write here]
```

---

### Real Brain Examples: Putting Theory into Practice

The E-I feedback motif is a "universal" building block. Now that you have modeled it, look at these three biological examples.

**Challenge:** Based on your simulation results, match the brain area to the specific functional role you think its inhibitory feedback is performing.

| Brain Area | Proposed Functional Role [Put number that matches function] |
|:---|:---|
| **1. Cortex** (Pyramidal cells + Interneurons) | [ ] Sharpening signals by suppressing the "weakest" inputs. |
| **2. Cerebellum** (Granule cells + Golgi cells) | [ ] Acting as a safety brake to prevent runaway excitation. |
| **3. Olfactory Bulb** (Mitral cells + Granule cells) | [ ] Acting as a timer to limit how long a signal lasts. |


*In your own words, why is a purely excitatory brain (one with no inhibitory motifs) physically unable to function? What did you observe in the simulation when `J_ei` was set to 0 that supports your answer?*

**My Answer:**
```
[Write here]
```

---

### Checkpoint 1

- When E drives I and I inhibits E, this is called: ______
- In the spike correlation plot, a peak at 0ms means: ______
- One biological function of feedback inhibition is: ______

## Part 2: An E-I-I Triplet — Adding Complexity

With just two neurons we saw how inhibitory feedback can regulate activity.  
But what happens when we add a *third* neuron?

In this motif:

- **Blue neuron (E)** — excitatory, receives external input  
- **Red neuron (I₁)** — inhibitory  
- **Purple neuron (I₂)** — inhibitory  

<center>
<img src="https://github.com/michaelglh/STG/blob/master/figs/EII.png?raw=1" alt="EII motif" width="300"/> </center>

The connections you can control are:

| Connection | Try to interpret this yourself |
|-----------|--------------------------------|
| **J_E1** | What effect does E have on I₁? <details><summary>Check</summary>E projects to I₁</details> |
| **J_E2** | What effect does E have on I₂? <details><summary>Check</summary>E projects to I₂</details> |
| **J_21** | What effect does I₂ have on I₁? <details><summary>Check</summary>I₂ inhibits I₁</details> |

> **Note:**  
> In this motif, all three connections have the same sign.  
>  
> Do you think they are positive or negative?
> <details><summary>Reveal</summary><strong>Negative (inhibitory)</strong></details>

---

### Predict

If I₂ inhibits I₁, and I₁ inhibits E, what do you predict happens to E when I₂ becomes active?

*Try to reason it through step by step rather than guessing.*

**My prediction:**

```
[Write here]
```

Sometimes, inhibiting an inhibitory neuron can *increase* activity elsewhere in the network.

What do you think this type of interaction might do to overall network activity?

**My prediction:**
```
[Write here]
```

This pattern has a specific name:

<details><summary>Click to reveal</summary>
<strong>Disinhibition</strong> — inhibiting an inhibitory neuron can increase activity in the circuit.
</details>

Do you think adding more inhibitory neurons always reduces overall activity in the network? Why or why not?

**My prediction:**
```
[Write here]
```


The external input to each neuron can be:

- **Icur** — steady current with mild noise  
- **Gaussian** — continuous random fluctuations  
- **Poisson** — discrete, spike-like random events  

How do you predict the network’s activity will differ between these three input types?

- Which one will produce the most **irregular** firing?  
- Which one will produce the most **stable** firing?  
- In which case might the effect of inhibition (or disinhibition) be harder to interpret?

**My prediction:**
```
[Write here]
```


<!-- Anatomical Divegent and Convergent

Functional Inhibitory;Disinhibitory -->

In [3]:
#@title Run the following start E-I-I triplet simulation { vertical-output: true }

# Triplet
N = 3                  # number of neurons
wt, dl = 1., 5.
rt = 65.0

# updating parameters
def update_EII(Itype='Icur', neuron_E='tonic_neuron', neuron_I1='tonic_neuron', neuron_I2='tonic_neuron',
               J_E1=0., J_E2=0., J_21=0.):

    # simulator
    h = Simulator(dt=dt)

    # network of neurons
    nrns = [LIF(sim=h) for _ in range(N)]
    nrns[0].update(neuron_params[neuron_E])
    nrns[1].update(neuron_params[neuron_I1])
    nrns[2].update(neuron_params[neuron_I2])

    # background noise
    if Itype == 'Icur':
        noises = [Current_injector(sim=h, rate=rt,
                                   start=int(T/dt*0.1),
                                   end=int(T/dt*0.9)) for _ in range(N)]
    elif Itype == 'Gaussian':
        noises = [Gaussian_generator(sim=h, mean=rt, std=rt) for _ in range(N)]
    elif Itype == 'Poisson':
        noises = [Poisson_generator(sim=h, rate=250) for _ in range(N)]
    else:
        print('Invalid input')

    for noise, nrn in zip(noises, nrns):
        nrn.connect(noise, {'ctype': 'Static', 'weight': wt, 'delay': dl})

    # recurrent connections
    con = np.array([[0., J_E1, J_E2],
                    [0., 0., 0.],
                    [0., J_21, 0.]])
    dly = np.random.uniform(2., 5., (N, N))
    synspecs = [[{'ctype': 'Static',
                  'weight': con[i, j],
                  'delay': dly[i, j]} for j in range(N)] for i in range(N)]
    h.connect(nrns, nrns, synspecs)

    # simulation
    h.run(T)

    # coincidence
    binwindow = int(5.0 / dt)
    spike_trains = [nrn.states for nrn in nrns]
    bin_spikes = [np.convolve(strain, np.ones(binwindow), 'same')
                  for strain in spike_trains]
    deltats = np.linspace(-10., 10., 21)
    coins_all = []

    for src, tar in zip([0, 0], [1, 2]):
        coins = []
        for delay in deltats:
            index = int(delay / dt)
            if index > 0:
                coins.append(np.dot(bin_spikes[src][:-index],
                                    bin_spikes[tar][index:]))
            elif index < 0:
                coins.append(np.dot(bin_spikes[tar][:index],
                                    bin_spikes[src][-index:]))
            else:
                coins.append(np.dot(bin_spikes[src], bin_spikes[tar]))
        coins = np.array(coins) / np.sqrt(np.dot(bin_spikes[src], bin_spikes[src])) \
                / np.sqrt(np.dot(bin_spikes[tar], bin_spikes[tar]))
        coins_all.append(coins)

    # --- Visualization ---
    plt.figure(figsize=(18, 14))   # bigger figure for readability

    # UPDATED COLORS: E = blue, I1 = red, I2 = lilac
    cs = ['#1f77b4', '#d62728', '#C77DFF']

    # Raster
    plt.subplot(3, N, 1)
    plt.title('Raster')
    for nrn, c, l in zip(nrns, cs, range(N)):
        plt.eventplot(nrn.spikes['times'], lineoffsets=2 * l, colors=c)
    plt.xlabel('Time (ms)')
    plt.yticks(np.array(range(N)) * 2, ['E', r'$I_1$', r'$I_2$'])
    plt.xlim([0., T])

    # Spike correlation
    plt.subplot(3, N, 2)
    plt.title('Spike correlation')
    plt.plot(deltats, coins_all[0], cs[1], label=r'$E-I_1$')
    plt.xlabel(r'$\delta t$ (ms)')
    plt.ylabel('corr')
    plt.ylim([0, 1])
    plt.legend()

    plt.subplot(3, N, 3)
    plt.plot(deltats, coins_all[1], cs[2], label=r'$E-I_2$')
    plt.xlabel(r'$\delta t$ (ms)')
    plt.ylabel('corr')
    plt.ylim([0, 1])
    plt.legend()

    # Voltage traces
    for idx, c in zip(range(N), cs):
        plt.subplot(3, N, idx + N + 1)
        if idx == 0:
            plt.title('Voltage traces')
        plt_par = {'dt': dt,
                   'range_t': np.arange(0., T, dt),
                   'V_th': nrns[idx].V_th}
        plot_volt_trace(plt_par, nrns[idx].v,
                        np.array(nrns[idx].spikes['times']), c=c)
        plt.xlim([0., T])

    # --- NEW: Instantaneous firing rate (bottom row) ---
    # This plot shows how active each neuron is over time.
    #
    # How to read it:
    # - The curve is a smoothed estimate of spikes per second (Hz)
    # - When the line goes UP → the neuron is firing more often
    # - When the line goes DOWN → the neuron is being inhibited
    #
    # Why this matters:
    # - You can SEE disinhibition: if I2 inhibits I1, the I1 curve drops
    #   and the E curve often rises shortly after.
    # - It summarizes the raster in one clean trace per neuron.
    # - It helps students understand how inhibition shapes firing rates.
    #
    # The smoothing window (50 ms) makes the curve readable.
    plt.subplot(3, 1, 3)
    plt.title("Instantaneous firing rate (Hz)")

    window = int(50/dt)  # 50 ms smoothing window
    time_axis = np.arange(0., T, dt)

    for nrn, c in zip(nrns, cs):
        spikes = nrn.states
        spikes = spikes[:len(time_axis)]  # ensure match
        rate = np.convolve(spikes, np.ones(window), 'same') * (1000/window)
        plt.plot(time_axis, rate, color=c)

    plt.xlabel("Time (ms)")
    plt.ylabel("Rate (Hz)")
    plt.xlim([0, T])

    plt.tight_layout()
    plt.show()


# No initial figure created
widgets.interact(
    update_EII,
    Itype=['Icur', 'Gaussian', 'Poisson'],
    neuron_E=neuron_params.keys(),
    neuron_I1=neuron_params.keys(),
    neuron_I2=neuron_params.keys(),
    J_E1=(-50., 0., 0.1),
    J_E2=(-50., 0., 0.1),
    J_21=(-50., 0., 0.1)
);


interactive(children=(Dropdown(description='Itype', options=('Icur', 'Gaussian', 'Poisson'), value='Icur'), Dr…

### Explain — E-I-I Triplet

**Start with all connections at 0**

*Describe the baseline activity of all three neurons. What do the instantaneous frequency traces look like when there are no connections?*

**My answer:**
```
[Write here]
```

---

**Connect E to I₁ only (J_E1 negative, others at 0)**

*What happens to I₁'s firing rate? What happens to E? Look at the instantaneous frequency plot — does I₁'s frequency track E's activity?*

**My answer:**
```
[Write here]
```

---

**Now also connect E to I₂ (J_E2 negative)**

*Both inhibitory neurons are now driven by E. How does E's instantaneous frequency change compared to when only I₁ was connected?*

**My answer:**
```
[Write here]
```

---

**Now connect I₂ back to I₁ (J_21 negative)**

*This creates a disinhibitory pathway: E → I₂ → inhibits I₁ → releases E. What do you observe in the instantaneous frequency traces? Does I₁'s activity change?*

**My answer:**
```
[Write here]
```

*Does inhibiting an inhibitory neuron always increase overall network activity? What does it depend on?*

**My answer:**
```
[Write here]
```
markdown

#### How Input Type Shapes Your Interpretation of a Motif

**1. Test each input type with a simple motif (E → I₁ only)**

Set:
- J_E1 = a moderate negative value (e.g., –20)  
- J_E2 = 0  
- J_21 = 0  

Then switch between **Icur**, **Gaussian**, and **Poisson**.

*How does the input type change your perception of the motif?*

Guiding ideas:
- Does Icur make the inhibition look smoother and more “continuous”?  
- Does Gaussian input make the motif look more unstable or noisy?  
- Does Poisson input make the inhibitory effect look more abrupt or burst‑like?  
- Does the instantaneous frequency plot make the motif look stronger or weaker depending on the input?

**My answer:**

```
[Write here]
```

---

**2. Test a full disinhibition motif under different inputs**

Set:
- J_E1 = negative  
- J_E2 = negative  
- J_21 = strong negative (e.g., –40 to –50)  

This creates:  
E → I₂ → inhibits I₁ → releases E

Now switch between **Icur**, **Gaussian**, and **Poisson**.

*Does the input type change how obvious the disinhibition effect is?*

Guiding ideas:
- Under Icur, does the disinhibition look smooth and gradual?  
- Under Gaussian, does the release of E look more variable or jittery?  
- Under Poisson, does disinhibition appear as sudden jumps in E’s firing rate?  
- Does the input type ever make the motif look *weaker* or *stronger* than it really is?

**My answer:**

```
[Write here]
```
---

## Circuit Challenges

For each circuit state below, try to find the connection weights that produce it. Use the **raster**, **instantaneous frequency**, and **spike correlation** plots together to judge whether you've succeeded.

Write down the weight values that worked and describe what you see in each plot.

---

### Challenge 1 — "Pure inhibition"

*Find a configuration where one inhibitory neuron strongly and consistently suppresses another neuron's activity. The instantaneous frequency of the target should drop noticeably or reach zero.*

**Weights I used:**
```
J_E1 =
J_E2 =
J_21 =
```

**What I observed:**
```
[Write here]
```

<details>
<summary>💡 Hint 1 — click if stuck</summary>

Think about which connection directly targets which neuron. You only need one connection active to achieve this. What does "pure" inhibition imply about the number of pathways involved?

</details>

<details>
<summary>💡 Hint 2 — click if still stuck</summary>

Focus on a single strong negative weight. Set the others to zero. Watch the instantaneous frequency of the target neuron — does it fall to near zero? Try increasing the magnitude gradually rather than jumping straight to a large value.

</details>

---

### Challenge 2 — "Strong disinhibition"

*Find a configuration where inhibiting an inhibitory neuron causes a third neuron to fire more than it does with no connections at all. The instantaneous frequency of the released neuron should be clearly elevated.*

**Weights I used:**
```
J_E1 =
J_E2 =
J_21 =
```

**What I observed:**
```
[Write here]
```

<details>
<summary>💡 Hint 1 — click if stuck</summary>

Disinhibition requires at least two connections working in sequence. Think about the chain: which neuron needs to be suppressed so that another neuron can be released? Sketch the pathway on paper before adjusting the sliders.

</details>

<details>
<summary>💡 Hint 2 — click if still stuck</summary>

You need both J_E1 and J_21 active. Consider the logic: E drives I₂, and I₂ inhibits I₁. If I₁ was inhibiting something, what happens when I₁ goes quiet? Check the instantaneous frequency of E and I₁ separately.

</details>

---

### Challenge 3 — "Weak E drive"

*Find a configuration where E drives both inhibitory neurons but only weakly — the inhibitory neurons fire occasionally but E is not strongly suppressed. All three neurons should be visible and active in the raster, and the instantaneous frequencies should all be moderate and relatively stable.*

**Weights I used:**
```
J_E1 =
J_E2 =
J_21 =
```

**What I observed:**
```
[Write here]
```

<details>
<summary>💡 Hint 1 — click if stuck</summary>

The word "weak" is important here. Think about what happens to network activity as you gradually increase connection strength from zero — there is a range before effects become dramatic. Where is that range?

</details>

<details>
<summary>💡 Hint 2 — click if still stuck</summary>

Try setting both J_E1 and J_E2 to small negative values (close to 0) rather than large ones. Look at the instantaneous frequency plot — you are aiming for all three lines to be present and active, with none reaching zero.

</details>

---

### Challenge 4 — "Balanced network"

*This is the hardest challenge. Find a configuration where all three neurons fire at similar rates and the spike correlation between E and both inhibitory neurons is low — meaning they are active but not strongly locked to each other. The instantaneous frequency traces should fluctuate without any one neuron dominating.*

**Weights I used:**
```
J_E1 =
J_E2 =
J_21 =
```

**What I observed:**
```
[Write here]
```

<details>
<summary>💡 Hint 1 — click if stuck</summary>

A "balanced" network implies that excitation and inhibition are roughly offsetting each other. Think about what input type might help introduce variability that prevents rigid synchrony. Is Icur the right choice here?

</details>

<details>
<summary>💡 Hint 2 — click if still stuck</summary>

Try switching to Poisson or Gaussian input — this introduces noise that can decorrelate neurons. Then tune the inhibitory weights so they are present but not overwhelming. Watch the spike correlation plots: you are aiming for them to be relatively flat rather than sharply peaked.

</details>

---

### Final Checkpoint

- Disinhibition requires at least ______ connections in sequence
- In the "balanced" challenge, high spike correlation would suggest: ______
- The instantaneous frequency plot is useful because it shows: ______

---

### Bonus — Think deeper

*We have now seen single neurons, pairs, and triplets. As a network grows larger, do you think it becomes easier or harder to predict its behaviour from the properties of its individual neurons and connections? What does this imply for how we study neural circuits in the brain?*

**My answer:**
```
[Write here]
```
